# 01 - Hello OpenAI

This workbook shows you how to use OpenAI API in Python. 

In [1]:
import os
from rich.console import Console
from rich.markdown import Markdown
from dotenv import load_dotenv
from openai import OpenAI

In [2]:
load_dotenv(override=True)
console = Console()

In [3]:
# create an instance of our LLM
openai = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

Our very first interaction with OpenAI

In [4]:
response = openai.chat.completions.create(
    model="gpt-4o-mini",
    temperature=0.5,
    messages=[
        {
            "role": "user",
            "content": "Hello OpenAI, this is my first ever message to you.",
        }
    ],
)
console.print("[sky_blue1]OpenAI response to first message:[/sky_blue1]")
print(response.choices[0].message.content)

OpenAI response to first message:

Hello! Welcome! I'm glad to hear from you. How can I assist you today?


In [5]:
# now ask it to tell you a fun fact
response = openai.chat.completions.create(
    model="gpt-4o-mini",
    temperature=0.75,
    messages=[
        {
            "role": "user",
            "content": "Tell me a fun fact other than honey",
        }
    ],
)
console.print("[sky_blue1]OpenAI response to first message:[/sky_blue1]")
print(response.choices[0].message.content)

OpenAI response to first message:

Did you know that octopuses have three hearts? Two of the hearts pump blood to the gills, where it gets oxygenated, while the third heart pumps the oxygen-rich blood to the rest of the body. Interestingly, when an octopus swims, the heart that delivers blood to the body actually stops beating, which is why they prefer crawling over swimming!


This is a more practical example, where we define a _System Prompt_ as well as a _User Prompt_

Let's feed the model with a user message like the one above, which _contains_ instructions. It should generate a tutorial as a bulleted list as instructed in the SYSTEM_PROMPT.

In [6]:
# lets define a convenience function
def ask_openai(
    user_message: str, system_prompt: None, temperature=0.5, model="gpt-4o-mini"
) -> str:
    """responds to a user_message (with an optional system message) from an OpenAI LLM"""
    # define a default system prompt if the user does not provide one
    OUR_SYSTEM_PROMPT = (
        "You are a helpful assistant" if system_prompt is None else system_prompt
    )
    # ask our LLM
    response = openai.chat.completions.create(
        model=model,
        temperature=temperature,
        messages=[
            {"role": "system", "content": OUR_SYSTEM_PROMPT},
            {"role": "user", "content": user_message},
        ],
    )
    return response.choices[0].message.content

In [7]:
SYSTEM_PROMPT = """
You are an AI assistant that helps humans by generating tutorials given a text.
You will be provided with a text. If the text contains any kind of
instructions on how to proceed with something, generate a tutorial in a
bullet list with markdown formatting.
Otherwise, inform the user that the text does not contain any instructions.

Text:
"""

In [8]:
USER_MESSAGE = """
To prepare the known sauce from Genova, Italy, you can start by toasting
the pine nuts to then coarsely chop them in a kitchen mortar together with 
basil and garlic. Then, add half of the oil in the kitchen mortar and 
season with salt and pepper.
Finally, transfer the pesto to a bowl and stir in the grated Parmesan
cheese.
"""

In [9]:
# OpenAI should generate a tutorial as a bulleted list for this user prompt
console.print("[sky_blue1]OpenAI response:[/sky_blue1]")
print(ask_openai(user_message=USER_MESSAGE, system_prompt=SYSTEM_PROMPT))

OpenAI response:

# Tutorial: How to Prepare Genovese Pesto Sauce

1. **Toast the Pine Nuts**
   - Place pine nuts in a dry skillet over medium heat.
   - Toast them until they are golden brown, stirring frequently to avoid burning.

2. **Prepare the Ingredients**
   - Coarsely chop the toasted pine nuts using a kitchen mortar.
   - Add fresh basil leaves and garlic to the mortar.

3. **Combine Ingredients**
   - Use the pestle to grind the basil, garlic, and pine nuts together until you achieve a coarse paste.

4. **Add Oil and Seasoning**
   - Pour in half of the olive oil into the mortar.
   - Season the mixture with salt and pepper to taste.

5. **Transfer to a Bowl**
   - Carefully transfer the pesto mixture to a bowl.

6. **Add Cheese**
   - Stir in grated Parmesan cheese to the pesto mixture until well combined.

7. **Serve and Enjoy**
   - Use the pesto sauce immediately with pasta, bread, or as a condiment for various dishes.


In [10]:
# this user message does NOT contain instructions
USER_MESSAGE2 = "The sun is shining and the birds are chirping"

# but not for this one
console.print("[sky_blue1]OpenAI response:[/sky_blue1]")
print(ask_openai(user_message=USER_MESSAGE2, system_prompt=SYSTEM_PROMPT))

OpenAI response:

The text does not contain any instructions.


## Summarizing web pages
Here we'll try & fetch the contents of a web page (URL) and ask OpenAI to summarize the contents.
We have defined a convenience function `scrape_webpage()` in `scraper.py`

In [11]:
from scraper import scrape_webpage

URL = "https://edwarddonner.com"
scraped_text = scrape_webpage(URL)
print(scraped_text)

Home - Edward Donner

Home
AI Curriculum
Proficient AI Engineer
Connect Four
Outsmart
An arena that pits LLMs against each other in a battle of diplomacy and deviousness
About
Posts
Well, hi there.
I’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (
very
amateur) and losing myself in
Hacker News
, nodding my head sagely to things I only half understand.
I’m the co-founder and CTO of
Nebula.io
. We’re applying AI to a field where it can make a massive, positive impact: helping people discover their potential and pursue their reason for being. I’m previously the founder and CEO of AI startup untapt,
acquired in 2021
.
I will happily drone on for hours about LLMs to anyone in my vicinity. My friends got fed up with my impromptu lectures, and convinced me to make some Udemy courses. To my total joy (and shock) they’ve become best-selling, top-rated courses, with 500,000 enrolled across 194

In [12]:
system_prompt = """ 
You are a helpful assistant that analyzes the contents of a webpage
and provides a short summary, ignoring text that might be navigation related.
Respond in markdown.
"""

user_message = f"""Here is the text scraped from a webpage: 
{scrape_webpage("https://edwarddonner.com")}
If it contains news or announcements, them symmarize those too"""

response = ask_openai(user_message=user_message, system_prompt=system_prompt)
console.print("[sky_blue1]OpenAI response:[/sky_blue1]")
console.print(Markdown(response))

OpenAI response:

Summary of Edward Donner's Webpage                                                                                 

Edward Donner is a co-founder and CTO of Nebula.io, where he focuses on leveraging AI to help individuals discover 
their potential. He has a background as the founder and CEO of the AI startup untapt, which was acquired in 2021.  
Ed enjoys coding, experimenting with large language models (LLMs), and producing amateur electronic music. He has  
created popular Udemy courses on LLMs, attracting 500,000 students from 194 countries.                             

Recent Posts                                                                                                       

 • February 17, 2026: Discusses resources for transitioning from "Vibe Coder" to "Agentic Engineer".               
 • January 4, 2026: Shares resources for building AI agents with n8n.                                              
 • September 15, 2025: Provides resources for deploying AI to production in the AI Engineering MLOps track.        
 • May 28, 2025: Offers guidance on the order of taking AI courses.                                                

Ed expresses gratitude towards visitors from his courses and is enthusiastic about sharing his knowledge on LLMs.